# ARC-AGI-3 LLM Agent (Qwen2.5-7B NF4 for T4)

LLM agent using Qwen2.5-7B-Instruct with NF4 quantization via bitsandbytes.
Designed for Tesla T4 GPU (15GB VRAM).


In [ ]:
import json
import os
import shutil
import subprocess
import sys
import time
from pathlib import Path

TRUE_SUBMISSION = os.environ.get("KAGGLE_IS_COMPETITION_RERUN", "").strip().lower() in {"1", "true", "yes"}
NOTEBOOK_START = time.time()
os.environ["MPLBACKEND"] = "Agg"
os.environ["PYDEVD_DISABLE_FILE_VALIDATION"] = "1"

WORKING = Path("/kaggle/working")
WORKING.mkdir(parents=True, exist_ok=True)

print(f"TRUE_SUBMISSION={TRUE_SUBMISSION}")
print("GPU check:")
subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,memory.free", "--format=csv,noheader"])

In [ ]:
# Install ARC runtime
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "--quiet",
    "--no-index", "--no-warn-conflicts", "--disable-pip-version-check",
    "--find-links", "/kaggle/input/competitions/arc-prize-2026-arc-agi-3/arc_agi_3_wheels",
    "arc-agi", "python-dotenv",
], stdout=subprocess.DEVNULL)
print("ARC runtime installed")

# Check bitsandbytes
try:
    import bitsandbytes
    print(f"bitsandbytes: {bitsandbytes.__version__}")
except ImportError:
    print("Installing bitsandbytes...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "bitsandbytes", "--quiet"])
    import bitsandbytes
    print(f"bitsandbytes installed: {bitsandbytes.__version__}")

In [ ]:
# Find model path
print("Searching for Qwen2.5-7B-Instruct...")
MODEL_PATH = None
for root, dirs, files in os.walk("/kaggle/input"):
    if "config.json" in files:
        config_path = os.path.join(root, "config.json")
        try:
            cfg = json.load(open(config_path))
            if cfg.get("model_type") in ("qwen2", "qwen"):
                MODEL_PATH = Path(root)
                print(f"Found Qwen2 model at: {MODEL_PATH}")
                break
        except Exception:
            pass

if MODEL_PATH is None:
    for p in sorted(Path("/kaggle/input").iterdir()):
        if p.is_dir() and (p / "config.json").exists():
            MODEL_PATH = p
            print(f"Found model at: {MODEL_PATH}")
            break

if MODEL_PATH is None:
    MODEL_PATH = Path("/kaggle/input/qwen2-5-7b-instruct")
    print(f"Fallback: {MODEL_PATH}")

print(f"Model path: {MODEL_PATH}")
print(f"Exists: {MODEL_PATH.exists()}")
if MODEL_PATH.exists():
    files = sorted(f.name for f in MODEL_PATH.iterdir())
    print(f"Files ({len(files)}): {files[:15]}")
    total = sum(f.stat().st_size for f in MODEL_PATH.iterdir() if f.is_file())
    print(f"Total: {total / 1e9:.1f} GB")

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(str(MODEL_PATH), trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("Configuring NF4...")
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)

print("Loading model with NF4...")
model = AutoModelForCausalLM.from_pretrained(
    str(MODEL_PATH),
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
    torch_dtype=torch.float16,
)
model.eval()

print("\nModel loaded! GPU memory:")
subprocess.run(["nvidia-smi", "--query-gpu=memory.used,memory.free", "--format=csv,noheader"])
print(f"Config: {model.config.num_hidden_layers} layers, {model.config.hidden_size} hidden")

# Smoke test
print("\nSmoke test...")
test_input = tokenizer("Hello!", return_tensors="pt").to(model.device)
with torch.no_grad():
    output = model.generate(**test_input, max_new_tokens=15, do_sample=False)
response = tokenizer.decode(output[0], skip_special_tokens=True)
print(f"Response: {response!r}")
print("\nMODEL READY!")

In [ ]:
# Deploy agent code (base64 encoded)
import base64
agent_b64 = "IiIiTExNIGFnZW50IGZvciBBUkMtQUdJLTMgdXNpbmcgUXdlbjIuNS03QiBvbiBUNC4iIiIKaW1wb3J0IGhhc2hsaWIKaW1wb3J0IGpzb24KaW1wb3J0IG9zCmltcG9ydCByYW5kb20KaW1wb3J0IHJlCmltcG9ydCB0aW1lCmltcG9ydCB0aHJlYWRpbmcKZnJvbSB0eXBpbmcgaW1wb3J0IEFueQoKZnJvbSBhcmNlbmdpbmUgaW1wb3J0IEFjdGlvbklucHV0LCBGcmFtZURhdGEsIEdhbWVBY3Rpb24sIEdhbWVTdGF0ZQpmcm9tIGFnZW50cy5hZ2VudCBpbXBvcnQgQWdlbnQKClJFU0VUID0gIlJFU0VUIgpDT01QTEVYX0FDVElPTiA9ICJBQ1RJT042IgpNT0RFTF9UT19FTkdJTkUgPSB7CiAgICAiVVAiOiAiQUNUSU9OMSIsICJET1dOIjogIkFDVElPTjIiLCAiTEVGVCI6ICJBQ1RJT04zIiwgIlJJR0hUIjogIkFDVElPTjQiLAogICAgIk1PVVNFIjogIkFDVElPTjYiLCAiQ0xJQ0siOiAiQUNUSU9ONiIsCn0KRU5HSU5FX1RPX01PREVMID0ge3Y6IGsgZm9yIGssIHYgaW4gTU9ERUxfVE9fRU5HSU5FLml0ZW1zKCl9CkVOR0lORV9UT19NT0RFTC51cGRhdGUoeyJBQ1RJT04xIjogIlVQIiwgIkFDVElPTjIiOiAiRE9XTiIsICJBQ1RJT04zIjogIkxFRlQiLCAiQUNUSU9ONCI6ICJSSUdIVCJ9KQoKQVJDX0NPTE9SX0NIQVJTID0gIld3Z0djQk1QUmJTWU9yTnAiCkFSQ19DT0xPUl9MRUdFTkQgPSAoCiAgICAiVz13aGl0ZSwgdz1saWdodCBncmF5LCBnPWdyYXksIEc9ZGFyayBncmF5LCBjPWNoYXJjb2FsLCBCPWJsYWNrLCAiCiAgICAiTT1tYWdlbnRhLCBQPXBpbmssIFI9cmVkLCBiPWJsdWUsIFM9c2t5IGJsdWUsIFk9eWVsbG93LCBPPW9yYW5nZSwgIgogICAgInI9ZGFyayByZWQsIE49bGlnaHQgZ3JlZW4sIHA9cHVycGxlIgopCgoKZGVmIGZvcm1hdF9ncmlkX2FzY2lpKGdyaWQpOgogICAgaWYgbm90IGdyaWQ6CiAgICAgICAgcmV0dXJuICIoZW1wdHkgZ3JpZCkiCiAgICBsaW5lcyA9IFtdCiAgICBmb3Igcm93IGluIGdyaWQ6CiAgICAgICAgY2hhcnMgPSBbQVJDX0NPTE9SX0NIQVJTW21heCgwLCBtaW4oMTUsIGludCh2KSkpXSBmb3IgdiBpbiByb3ddCiAgICAgICAgbGluZXMuYXBwZW5kKCIiLmpvaW4oY2hhcnMpKQogICAgcmV0dXJuICJcbiIuam9pbihsaW5lcykKCgpTWVNURU1fUFJPTVBUID0gKAogICAgIllvdSBhcmUgYW4gZXhwZXJ0IHB1enpsZSBzb2x2ZXIgcGxheWluZyBhIGdyaWQtYmFzZWQgZ2FtZS5cblxuIgogICAgZiJDT0xPUiBMRUdFTkQ6IHtBUkNfQ09MT1JfTEVHRU5EfVxuXG4iCiAgICAiVGhlIGdhbWUgYm9hcmQgaXMgYSA2NHg2NCBjb2xvciBncmlkLiBZb3UgdGFrZSBhY3Rpb25zIHRvIHNvbHZlIHB1enpsZXMuXG5cbiIKICAgICJBVkFJTEFCTEUgQUNUSU9OUzpcbiIKICAgICItIFVQL0RPV04vTEVGVC9SSUdIVDogbW92ZSBpbiBhIGRpcmVjdGlvblxuIgogICAgIi0gTU9VU0Uocm93PU4sY29sPU0pOiBjbGljayBhdCBwb3NpdGlvblxuXG4iCiAgICAiRWFjaCB0dXJuLCBvdXRwdXQgYSBKU09OIGFjdGlvbiBvbiB0aGUgTEFTVCBsaW5lOlxuIgogICAgJ2BgYGpzb25cbnsiYWN0aW9uIjogIlVQIiwgInJlYXNvbmluZyI6ICJicmllZiBleHBsYW5hdGlvbiJ9XG5gYGBcbicKICAgICdGb3IgTU9VU0U6IHsiYWN0aW9uIjogIk1PVVNFIiwgInJvdyI6IDUsICJjb2wiOiAxMCwgInJlYXNvbmluZyI6ICIuLi4ifVxuXG4nCiAgICAiU1RSQVRFR0lFUzpcbiIKICAgICItIEV4cGxvcmUgYnkgbW92aW5nIGluIGRpZmZlcmVudCBkaXJlY3Rpb25zXG4iCiAgICAiLSBJZGVudGlmeSB0aGUgcGxheWVyIGF2YXRhciAoZGlzdGluY3QgY29sb3JlZCBzaGFwZSB0aGF0IG1vdmVzKVxuIgogICAgIi0gRmluZCBvYnN0YWNsZXMsIHRhcmdldHMsIGFuZCBpbnRlcmFjdGl2ZSBvYmplY3RzXG4iCiAgICAiLSBGb3IgbmF2aWdhdGlvbiwgcGxhbiBwYXRocyBtZW50YWxseVxuIgogICAgIi0gSWYgc3R1Y2sgZm9yIDEwKyBhY3Rpb25zLCBjaGFuZ2Ugc3RyYXRlZ3kgcmFkaWNhbGx5XG4iCiAgICAiLSBEbyBub3QgY2xpY2sgb24gSFVEIGVsZW1lbnRzIChiYXJzIGF0IGVkZ2VzKVxuIgogICAgIi0gT2JqZWN0cyBhcmUgbXVsdGktdGlsZSBjb25uZWN0ZWQgY29tcG9uZW50cyBvZiB0aGUgc2FtZSBjb2xvclxuXG4iCiAgICAiQmUgZGVjaXNpdmUuIEFsd2F5cyBvdXRwdXQgZXhhY3RseSBvbmUgSlNPTiBhY3Rpb24gYmxvY2suIgopCgoKZGVmIGV4dHJhY3RfZ3JpZChmcmFtZV9kYXRhKToKICAgIGZvciBhdHRyX2ZuIGluIFsKICAgICAgICBsYW1iZGEgZjogZi5yYXdfc3RhdGUuZGF0YSwKICAgICAgICBsYW1iZGEgZjogZi5yYXdfc3RhdGUucGxhbmVzWzBdIGlmIGYucmF3X3N0YXRlLnBsYW5lcyBlbHNlIE5vbmUsCiAgICAgICAgbGFtYmRhIGY6IGYucGxhbmVzWzBdIGlmIGhhc2F0dHIoZiwgInBsYW5lcyIpIGFuZCBmLnBsYW5lcyBlbHNlIE5vbmUsCiAgICBdOgogICAgICAgIHRyeToKICAgICAgICAgICAgZGF0YSA9IGF0dHJfZm4oZnJhbWVfZGF0YSkKICAgICAgICAgICAgaWYgZGF0YSBpcyBOb25lOgogICAgICAgICAgICAgICAgY29udGludWUKICAgICAgICAgICAgaWYgaGFzYXR0cihkYXRhLCAidG9saXN0Iik6CiAgICAgICAgICAgICAgICByb3dzID0gZGF0YS50b2xpc3QoKQogICAgICAgICAgICBlbHNlOgogICAgICAgICAgICAgICAgcm93cyA9IGxpc3QoZGF0YSkKICAgICAgICAgICAgaWYgcm93cyBhbmQgcm93c1swXToKICAgICAgICAgICAgICAgIHJldHVybiB0dXBsZSh0dXBsZShpbnQoYykgZm9yIGMgaW4gcm93KSBmb3Igcm93IGluIHJvd3MpCiAgICAgICAgZXhjZXB0IEV4Y2VwdGlvbjoKICAgICAgICAgICAgY29udGludWUKICAgIHJldHVybiAoKDAsKSAqIDY0LCkgKiA2NAoKCmRlZiBmcmFtZV90b19pbmZvKGZyYW1lX2RhdGEpOgogICAgZ3JpZCA9IGV4dHJhY3RfZ3JpZChmcmFtZV9kYXRhKQogICAgdHJ5OgogICAgICAgIHN0YXRlID0gZnJhbWVfZGF0YS5zdGF0ZQogICAgICAgIHN0YXRlX25hbWUgPSBzdGF0ZS5uYW1lIGlmIGhhc2F0dHIoc3RhdGUsICJuYW1lIikgZWxzZSBzdHIoc3RhdGUpCiAgICBleGNlcHQgRXhjZXB0aW9uOgogICAgICAgIHN0YXRlX25hbWUgPSAiUExBWUlORyIKICAgIHRyeToKICAgICAgICBsZXZlbHNfY29tcGxldGVkID0gaW50KGZyYW1lX2RhdGEubGV2ZWxzX2NvbXBsZXRlZCkKICAgIGV4Y2VwdCBFeGNlcHRpb246CiAgICAgICAgbGV2ZWxzX2NvbXBsZXRlZCA9IDAKICAgIGF2YWlsYWJsZSA9IFtdCiAgICB0cnk6CiAgICAgICAgZm9yIGFjdF9pZCBpbiBmcmFtZV9kYXRhLmF2YWlsYWJsZV9hY3Rpb25zOgogICAgICAgICAgICB0cnk6CiAgICAgICAgICAgICAgICBhY3QgPSBHYW1lQWN0aW9uLmZyb21faWQoaW50KGFjdF9pZCkpCiAgICAgICAgICAgICAgICBpZiBhY3QubmFtZSAhPSAiUkVTRVQiOgogICAgICAgICAgICAgICAgICAgIGF2YWlsYWJsZS5hcHBlbmQoYWN0Lm5hbWUpCiAgICAgICAgICAgIGV4Y2VwdCBFeGNlcHRpb246CiAgICAgICAgICAgICAgICBwYXNzCiAgICBleGNlcHQgRXhjZXB0aW9uOgogICAgICAgIGF2YWlsYWJsZSA9IFsiQUNUSU9OMSIsICJBQ1RJT04yIiwgIkFDVElPTjMiLCAiQUNUSU9ONCIsICJBQ1RJT041IiwgIkFDVElPTjYiLCAiQUNUSU9ONyJdCiAgICB0cnk6CiAgICAgICAgc3RlcCA9IGludChmcmFtZV9kYXRhLnN0ZXApCiAgICBleGNlcHQgRXhjZXB0aW9uOgogICAgICAgIHN0ZXAgPSAwCiAgICByZXR1cm4gewogICAgICAgICJncmlkIjogZ3JpZCwgInN0ZXAiOiBzdGVwLCAibGV2ZWwiOiBsZXZlbHNfY29tcGxldGVkICsgMSwKICAgICAgICAic3RhdGVfbmFtZSI6IHN0YXRlX25hbWUsICJsZXZlbHNfY29tcGxldGVkIjogbGV2ZWxzX2NvbXBsZXRlZCwKICAgICAgICAiYXZhaWxhYmxlX2FjdGlvbnMiOiBhdmFpbGFibGUsCiAgICB9CgoKY2xhc3MgTExNQWdlbnQoQWdlbnQpOgogICAgX0FDVElPTl9MT0NLID0gdGhyZWFkaW5nLlJMb2NrKCkKICAgIEFDVElPTl9CVURHRVQgPSA0MDAKICAgIE1BWF9BQ1RJT05TID0gQUNUSU9OX0JVREdFVAogICAgR0FNRV9USU1FX0xJTUlUID0gMjQwCgogICAgX21vZGVsID0gTm9uZQogICAgX3Rva2VuaXplciA9IE5vbmUKICAgIF9jb252ZXJzYXRpb24gPSBbXQogICAgX2dhbWVfb3Zlcl9jb3VudCA9IDAKICAgIF90b3RhbF9hY3Rpb25zID0gMAogICAgX3VuaXF1ZV9zdGF0ZXMgPSBzZXQoKQogICAgX3JlY2VudF9zaWdzID0gW10KICAgIF9yZWNlbnRfYWN0aW9ucyA9IFtdCiAgICBfbGFzdF9sZXZlbCA9IDAKICAgIF9nYW1lX3N0YXJ0ID0gMAogICAgX2FjdGlvbnNfaW5fZ2FtZSA9IDAKICAgIF9maXJzdF9hY3Rpb24gPSBUcnVlCgogICAgQGNsYXNzbWV0aG9kCiAgICBkZWYgc2V0X21vZGVsKGNscywgbW9kZWwsIHRva2VuaXplcik6CiAgICAgICAgY2xzLl9tb2RlbCA9IG1vZGVsCiAgICAgICAgY2xzLl90b2tlbml6ZXIgPSB0b2tlbml6ZXIKCiAgICBkZWYgX19pbml0X18oc2VsZiwgKmFyZ3MsICoqa3dhcmdzKToKICAgICAgICBzdXBlcigpLl9faW5pdF9fKCphcmdzLCAqKmt3YXJncykKCiAgICBAcHJvcGVydHkKICAgIGRlZiBuYW1lKHNlbGYpOgogICAgICAgIHJldHVybiBmIntzdXBlcigpLm5hbWV9LmxsbS10NC12MSIKCiAgICBkZWYgaXNfZG9uZShzZWxmLCBmcmFtZXMsIGxhdGVzdF9mcmFtZSk6CiAgICAgICAgaWYgc2VsZi5fYWN0aW9uc19pbl9nYW1lID49IHNlbGYuQUNUSU9OX0JVREdFVDoKICAgICAgICAgICAgcmV0dXJuIFRydWUKICAgICAgICB0cnk6CiAgICAgICAgICAgIGlmIGxhdGVzdF9mcmFtZS5zdGF0ZSBpcyBHYW1lU3RhdGUuV0lOOgogICAgICAgICAgICAgICAgcmV0dXJuIFRydWUKICAgICAgICBleGNlcHQgRXhjZXB0aW9uOgogICAgICAgICAgICBwYXNzCiAgICAgICAgaWYgdGltZS50aW1lKCkgLSBzZWxmLl9nYW1lX3N0YXJ0ID4gc2VsZi5HQU1FX1RJTUVfTElNSVQ6CiAgICAgICAgICAgIHJldHVybiBUcnVlCiAgICAgICAgcmV0dXJuIEZhbHNlCgogICAgQHN0YXRpY21ldGhvZAogICAgZGVmIF90b19nYW1lX2FjdGlvbihwcm9wb3NhbCk6CiAgICAgICAgYWN0aW9uX25hbWUgPSBwcm9wb3NhbC5nZXQoIm5hbWUiLCAiUkVTRVQiKQogICAgICAgIHRyeToKICAgICAgICAgICAgYWN0aW9uID0gR2FtZUFjdGlvblthY3Rpb25fbmFtZV0KICAgICAgICBleGNlcHQgS2V5RXJyb3I6CiAgICAgICAgICAgIGFjdGlvbiA9IEdhbWVBY3Rpb24uUkVTRVQKICAgICAgICBpZiBhY3Rpb24uaXNfY29tcGxleCgpOgogICAgICAgICAgICB4ID0gcHJvcG9zYWwuZ2V0KCJ4IiwgMCkgb3IgMAogICAgICAgICAgICB5ID0gcHJvcG9zYWwuZ2V0KCJ5IiwgMCkgb3IgMAogICAgICAgICAgICBhY3Rpb24uc2V0X2RhdGEoeyJ4IjogbWF4KDAsIG1pbig2MywgaW50KHgpKSksICJ5IjogbWF4KDAsIG1pbig2MywgaW50KHkpKSl9KQogICAgICAgIGVsc2U6CiAgICAgICAgICAgIGFjdGlvbi5zZXRfZGF0YSh7fSkKICAgICAgICBhY3Rpb24ucmVhc29uaW5nID0gcHJvcG9zYWwuZ2V0KCJyZWFzb25pbmciLCB7fSkKICAgICAgICByZXR1cm4gYWN0aW9uCgogICAgZGVmIF9zdGF0ZV9zaWcoc2VsZiwgZnJhbWVfaW5mbyk6CiAgICAgICAgZ3JpZCA9IGZyYW1lX2luZm9bImdyaWQiXQogICAgICAgIHNhbXBsZWQgPSBbXQogICAgICAgIGZvciByIGluIHJhbmdlKDAsIGxlbihncmlkKSwgNCk6CiAgICAgICAgICAgIHJvd19zID0gW2dyaWRbcl1bY10gZm9yIGMgaW4gcmFuZ2UoMCwgbGVuKGdyaWRbcl0pLCA0KV0KICAgICAgICAgICAgc2FtcGxlZC5hcHBlbmQodHVwbGUocm93X3MpKQogICAgICAgIHBheWxvYWQgPSBmJ3tmcmFtZV9pbmZvWyJsZXZlbCJdfTp7ZnJhbWVfaW5mb1sic3RhdGVfbmFtZSJdfTp7c2FtcGxlZH0nLmVuY29kZSgpCiAgICAgICAgcmV0dXJuIGhhc2hsaWIubWQ1KHBheWxvYWQpLmhleGRpZ2VzdCgpWzoxMl0KCiAgICBkZWYgX2lzX2xvb3BlZChzZWxmKToKICAgICAgICBpZiBsZW4oc2VsZi5fcmVjZW50X3NpZ3MpIDwgNjoKICAgICAgICAgICAgcmV0dXJuIEZhbHNlCiAgICAgICAgbGFzdDYgPSBzZWxmLl9yZWNlbnRfc2lnc1stNjpdCiAgICAgICAgaWYgbGFzdDZbMF0gPT0gbGFzdDZbMl0gPT0gbGFzdDZbNF0gYW5kIGxhc3Q2WzFdID09IGxhc3Q2WzNdID09IGxhc3Q2WzVdOgogICAgICAgICAgICByZXR1cm4gVHJ1ZQogICAgICAgIGlmIGxlbihzZWxmLl9yZWNlbnRfYWN0aW9ucykgPj0gNiBhbmQgbGVuKHNldChzZWxmLl9yZWNlbnRfYWN0aW9uc1stNjpdKSkgPT0gMToKICAgICAgICAgICAgcmV0dXJuIFRydWUKICAgICAgICBpZiBsZW4oc2VsZi5fcmVjZW50X3NpZ3MpID49IDQgYW5kIGxlbihzZXQoc2VsZi5fcmVjZW50X3NpZ3NbLTQ6XSkpID09IDE6CiAgICAgICAgICAgIHJldHVybiBUcnVlCiAgICAgICAgcmV0dXJuIEZhbHNlCgogICAgZGVmIF9idWlsZF91c2VyX3R1cm4oc2VsZiwgZnJhbWVfaW5mbyk6CiAgICAgICAgcGFydHMgPSBbCiAgICAgICAgICAgIGYnPT09IFR1cm4ge3NlbGYuX3RvdGFsX2FjdGlvbnMgKyAxfSB8IExldmVsIHtmcmFtZV9pbmZvWyJsZXZlbCJdfSB8ICcKICAgICAgICAgICAgZidTdGVwIHtmcmFtZV9pbmZvWyJzdGVwIl19IHwgU3RhdGU6IHtmcmFtZV9pbmZvWyJzdGF0ZV9uYW1lIl19ID09PScsCiAgICAgICAgICAgIGYnVmFsaWQgYWN0aW9uczogeyIsICIuam9pbihFTkdJTkVfVE9fTU9ERUwuZ2V0KGEsIGEpIGZvciBhIGluIGZyYW1lX2luZm9bImF2YWlsYWJsZV9hY3Rpb25zIl0pfScsCiAgICAgICAgXQogICAgICAgIGlmIHNlbGYuX3RvdGFsX2FjdGlvbnMgPiAwOgogICAgICAgICAgICBwYXJ0cy5hcHBlbmQoCiAgICAgICAgICAgICAgICBmIlN0YXRzOiB7c2VsZi5fdG90YWxfYWN0aW9uc30gYWN0aW9ucywge2xlbihzZWxmLl91bmlxdWVfc3RhdGVzKX0gdW5pcXVlIHN0YXRlcywgIgogICAgICAgICAgICAgICAgZiJ7c2VsZi5fZ2FtZV9vdmVyX2NvdW50fSBkZWF0aHMiCiAgICAgICAgICAgICkKICAgICAgICByb3dzID0gbGVuKGZyYW1lX2luZm9bImdyaWQiXSkKICAgICAgICBjb2xzID0gbGVuKGZyYW1lX2luZm9bImdyaWQiXVswXSkgaWYgZnJhbWVfaW5mb1siZ3JpZCJdIGVsc2UgMAogICAgICAgIGlmIHJvd3MgPD0gNDAgYW5kIGNvbHMgPD0gNDA6CiAgICAgICAgICAgIHBhcnRzLmFwcGVuZChmIlxuQm9hcmQgKHtyb3dzfXh7Y29sc30pOlxue2Zvcm1hdF9ncmlkX2FzY2lpKGZyYW1lX2luZm9bJ2dyaWQnXSl9IikKICAgICAgICBlbHNlOgogICAgICAgICAgICBwYXJ0cy5hcHBlbmQoZiJcbkJvYXJkIGlzIHtyb3dzfXh7Y29sc30gKGxhcmdlKSIpCiAgICAgICAgaWYgc2VsZi5fcmVjZW50X2FjdGlvbnM6CiAgICAgICAgICAgIHJlY2VudCA9IHNlbGYuX3JlY2VudF9hY3Rpb25zWy01Ol0KICAgICAgICAgICAgcGFydHMuYXBwZW5kKGYnXG5SZWNlbnQ6IHsiIC0+ICIuam9pbihFTkdJTkVfVE9fTU9ERUwuZ2V0KGEsIGEpIGZvciBhIGluIHJlY2VudCl9JykKICAgICAgICBpZiBzZWxmLl9pc19sb29wZWQoKToKICAgICAgICAgICAgcGFydHMuYXBwZW5kKCJcbldBUk5JTkc6IFN0dWNrIGluIGEgbG9vcCEgVHJ5IGEgQ09NUExFVEVMWSBkaWZmZXJlbnQgYXBwcm9hY2guIikKICAgICAgICByZXR1cm4gIlxuIi5qb2luKHBhcnRzKQoKICAgIGRlZiBfY2FsbF9sbG0oc2VsZiwgZnJhbWVfaW5mbyk6CiAgICAgICAgaWYgbm90IHNlbGYuX2NvbnZlcnNhdGlvbjoKICAgICAgICAgICAgc2VsZi5fY29udmVyc2F0aW9uLmFwcGVuZCh7InJvbGUiOiAic3lzdGVtIiwgImNvbnRlbnQiOiBTWVNURU1fUFJPTVBUfSkKCiAgICAgICAgdXNlcl9jb250ZW50ID0gc2VsZi5fYnVpbGRfdXNlcl90dXJuKGZyYW1lX2luZm8pCiAgICAgICAgc2VsZi5fY29udmVyc2F0aW9uLmFwcGVuZCh7InJvbGUiOiAidXNlciIsICJjb250ZW50IjogdXNlcl9jb250ZW50fSkKCiAgICAgICAgaWYgbGVuKHNlbGYuX2NvbnZlcnNhdGlvbikgPiAyMDoKICAgICAgICAgICAgc3lzdGVtID0gW20gZm9yIG0gaW4gc2VsZi5fY29udmVyc2F0aW9uIGlmIG1bInJvbGUiXSA9PSAic3lzdGVtIl0KICAgICAgICAgICAgb3RoZXJzID0gW20gZm9yIG0gaW4gc2VsZi5fY29udmVyc2F0aW9uIGlmIG1bInJvbGUiXSAhPSAic3lzdGVtIl0KICAgICAgICAgICAgc2VsZi5fY29udmVyc2F0aW9uID0gc3lzdGVtICsgb3RoZXJzWy0xNjpdCgogICAgICAgIHRleHQgPSBzZWxmLl90b2tlbml6ZXIuYXBwbHlfY2hhdF90ZW1wbGF0ZSgKICAgICAgICAgICAgc2VsZi5fY29udmVyc2F0aW9uLCB0b2tlbml6ZT1GYWxzZSwgYWRkX2dlbmVyYXRpb25fcHJvbXB0PVRydWUKICAgICAgICApCiAgICAgICAgaW5wdXRzID0gc2VsZi5fdG9rZW5pemVyKHRleHQsIHJldHVybl90ZW5zb3JzPSJwdCIpLnRvKHNlbGYuX21vZGVsLmRldmljZSkKCiAgICAgICAgd2l0aCB0b3JjaC5ub19ncmFkKCk6CiAgICAgICAgICAgIG91dHB1dHMgPSBzZWxmLl9tb2RlbC5nZW5lcmF0ZSgKICAgICAgICAgICAgICAgICoqaW5wdXRzLCBtYXhfbmV3X3Rva2Vucz0yNTYsIGRvX3NhbXBsZT1UcnVlLAogICAgICAgICAgICAgICAgdGVtcGVyYXR1cmU9MC42LCB0b3BfcD0wLjk1LCBwYWRfdG9rZW5faWQ9c2VsZi5fdG9rZW5pemVyLmVvc190b2tlbl9pZCwKICAgICAgICAgICAgKQoKICAgICAgICBuZXdfdG9rZW5zID0gb3V0cHV0c1swXVtpbnB1dHNbImlucHV0X2lkcyJdLnNoYXBlWzFdOl0KICAgICAgICByZXNwb25zZSA9IHNlbGYuX3Rva2VuaXplci5kZWNvZGUobmV3X3Rva2Vucywgc2tpcF9zcGVjaWFsX3Rva2Vucz1UcnVlKQogICAgICAgIHNlbGYuX2NvbnZlcnNhdGlvbi5hcHBlbmQoeyJyb2xlIjogImFzc2lzdGFudCIsICJjb250ZW50IjogcmVzcG9uc2V9KQoKICAgICAgICByZXR1cm4gc2VsZi5fcGFyc2VfcmVzcG9uc2UocmVzcG9uc2UsIGZyYW1lX2luZm8pCgogICAgZGVmIF9wYXJzZV9yZXNwb25zZShzZWxmLCB0ZXh0LCBmcmFtZV9pbmZvKToKICAgICAgICBqc29uX2Jsb2NrcyA9IHJlLmZpbmRhbGwocidgYGBqc29uXHMqKFx7Lio/XH0pXHMqYGBgJywgdGV4dCwgcmUuRE9UQUxMKQogICAgICAgIGlmIG5vdCBqc29uX2Jsb2NrczoKICAgICAgICAgICAganNvbl9ibG9ja3MgPSByZS5maW5kYWxsKHInXHtbXnt9XSoiYWN0aW9uIltee31dKlx9JywgdGV4dCkKCiAgICAgICAgZm9yIGJsb2NrIGluIGpzb25fYmxvY2tzOgogICAgICAgICAgICB0cnk6CiAgICAgICAgICAgICAgICBwYXJzZWQgPSBqc29uLmxvYWRzKGJsb2NrKQogICAgICAgICAgICAgICAgYWN0aW9uX25hbWUgPSBzdHIocGFyc2VkLmdldCgiYWN0aW9uIiwgIiIpKS51cHBlcigpLnN0cmlwKCkKICAgICAgICAgICAgICAgIGlmIG5vdCBhY3Rpb25fbmFtZToKICAgICAgICAgICAgICAgICAgICBjb250aW51ZQogICAgICAgICAgICAgICAgZW5naW5lX25hbWUgPSBNT0RFTF9UT19FTkdJTkUuZ2V0KGFjdGlvbl9uYW1lLCBhY3Rpb25fbmFtZSkKICAgICAgICAgICAgICAgIHZhbGlkID0gc2V0KGZyYW1lX2luZm9bImF2YWlsYWJsZV9hY3Rpb25zIl0pIHwgeyJSRVNFVCJ9CiAgICAgICAgICAgICAgICBpZiBlbmdpbmVfbmFtZSBub3QgaW4gdmFsaWQ6CiAgICAgICAgICAgICAgICAgICAgZm9yIGF2YWlsIGluIGZyYW1lX2luZm9bImF2YWlsYWJsZV9hY3Rpb25zIl06CiAgICAgICAgICAgICAgICAgICAgICAgIGlmIGF2YWlsIGluIGFjdGlvbl9uYW1lIG9yIGFjdGlvbl9uYW1lIGluIGF2YWlsOgogICAgICAgICAgICAgICAgICAgICAgICAgICAgZW5naW5lX25hbWUgPSBhdmFpbAogICAgICAgICAgICAgICAgICAgICAgICAgICAgYnJlYWsKICAgICAgICAgICAgICAgICAgICBlbHNlOgogICAgICAgICAgICAgICAgICAgICAgICBjb250aW51ZQogICAgICAgICAgICAgICAgcmVzdWx0ID0geyJuYW1lIjogZW5naW5lX25hbWUsICJyZWFzb25pbmciOiB7fX0KICAgICAgICAgICAgICAgIGlmIGVuZ2luZV9uYW1lID09IENPTVBMRVhfQUNUSU9OOgogICAgICAgICAgICAgICAgICAgIHJvdyA9IHBhcnNlZC5nZXQoInJvdyIsIHBhcnNlZC5nZXQoInkiLCAwKSkKICAgICAgICAgICAgICAgICAgICBjb2wgPSBwYXJzZWQuZ2V0KCJjb2wiLCBwYXJzZWQuZ2V0KCJ4IiwgMCkpCiAgICAgICAgICAgICAgICAgICAgcmVzdWx0WyJ4Il0gPSBpbnQoY29sKQogICAgICAgICAgICAgICAgICAgIHJlc3VsdFsieSJdID0gaW50KHJvdykKICAgICAgICAgICAgICAgIHJlYXNvbmluZyA9IHBhcnNlZC5nZXQoInJlYXNvbmluZyIsICIiKQogICAgICAgICAgICAgICAgaWYgcmVhc29uaW5nOgogICAgICAgICAgICAgICAgICAgIHJlc3VsdFsicmVhc29uaW5nIl1bImxsbSJdID0gc3RyKHJlYXNvbmluZylbOjIwMF0KICAgICAgICAgICAgICAgIHJldHVybiByZXN1bHQKICAgICAgICAgICAgZXhjZXB0IChqc29uLkpTT05EZWNvZGVFcnJvciwgVmFsdWVFcnJvcik6CiAgICAgICAgICAgICAgICBjb250aW51ZQoKICAgICAgICB0ZXh0X3VwcGVyID0gdGV4dC51cHBlcigpCiAgICAgICAgZm9yIHBhdHRlcm4sIGVuZ2luZV9uYW1lIGluIFsKICAgICAgICAgICAgKHInXGJVUFxiJywgIkFDVElPTjEiKSwgKHInXGJET1dOXGInLCAiQUNUSU9OMiIpLAogICAgICAgICAgICAocidcYkxFRlRcYicsICJBQ1RJT04zIiksIChyJ1xiUklHSFRcYicsICJBQ1RJT040IiksCiAgICAgICAgICAgIChyJ1xiUkVTRVRcYicsICJSRVNFVCIpLAogICAgICAgIF06CiAgICAgICAgICAgIGlmIHJlLnNlYXJjaChwYXR0ZXJuLCB0ZXh0X3VwcGVyKToKICAgICAgICAgICAgICAgIGlmIGVuZ2luZV9uYW1lIGluIGZyYW1lX2luZm9bImF2YWlsYWJsZV9hY3Rpb25zIl0gb3IgZW5naW5lX25hbWUgPT0gIlJFU0VUIjoKICAgICAgICAgICAgICAgICAgICByZXR1cm4geyJuYW1lIjogZW5naW5lX25hbWUsICJyZWFzb25pbmciOiB7ImZyb21fdGV4dCI6IFRydWV9fQogICAgICAgIHJldHVybiB7Im5hbWUiOiAiUkVTRVQiLCAicmVhc29uaW5nIjogeyJwYXJzZV9mYWlsZWQiOiBUcnVlfX0KCiAgICBkZWYgY2hvb3NlX2FjdGlvbihzZWxmLCBmcmFtZXMsIGxhdGVzdF9mcmFtZSk6CiAgICAgICAgc2VsZi5fQUNUSU9OX0xPQ0suYWNxdWlyZSgpCiAgICAgICAgdHJ5OgogICAgICAgICAgICBmcmFtZV9pbmZvID0gZnJhbWVfdG9faW5mbyhsYXRlc3RfZnJhbWUpCgogICAgICAgICAgICBpZiBzZWxmLl9maXJzdF9hY3Rpb246CiAgICAgICAgICAgICAgICBzZWxmLl9maXJzdF9hY3Rpb24gPSBGYWxzZQogICAgICAgICAgICAgICAgc2VsZi5fZ2FtZV9zdGFydCA9IHRpbWUudGltZSgpCiAgICAgICAgICAgICAgICBzZWxmLl9nYW1lX292ZXJfY291bnQgPSAwCiAgICAgICAgICAgICAgICBzZWxmLl90b3RhbF9hY3Rpb25zID0gMAogICAgICAgICAgICAgICAgc2VsZi5fdW5pcXVlX3N0YXRlcy5jbGVhcigpCiAgICAgICAgICAgICAgICBzZWxmLl9yZWNlbnRfc2lncy5jbGVhcigpCiAgICAgICAgICAgICAgICBzZWxmLl9yZWNlbnRfYWN0aW9ucy5jbGVhcigpCiAgICAgICAgICAgICAgICBzZWxmLl9jb252ZXJzYXRpb24uY2xlYXIoKQogICAgICAgICAgICAgICAgc2VsZi5fbGFzdF9sZXZlbCA9IDAKICAgICAgICAgICAgICAgIHNlbGYuX2FjdGlvbnNfaW5fZ2FtZSA9IDAKCiAgICAgICAgICAgIGlmIGZyYW1lX2luZm9bInN0YXRlX25hbWUiXSA9PSAiR0FNRV9PVkVSIjoKICAgICAgICAgICAgICAgIHNlbGYuX2dhbWVfb3Zlcl9jb3VudCArPSAxCiAgICAgICAgICAgICAgICBhY3Rpb24gPSBHYW1lQWN0aW9uLlJFU0VUCiAgICAgICAgICAgICAgICBhY3Rpb24uc2V0X2RhdGEoe30pCiAgICAgICAgICAgICAgICBhY3Rpb24ucmVhc29uaW5nID0geyJhdXRvX3Jlc2V0IjogVHJ1ZX0KICAgICAgICAgICAgICAgIHJldHVybiBhY3Rpb24KCiAgICAgICAgICAgIHNpZyA9IHNlbGYuX3N0YXRlX3NpZyhmcmFtZV9pbmZvKQogICAgICAgICAgICBzZWxmLl9yZWNlbnRfc2lncy5hcHBlbmQoc2lnKQogICAgICAgICAgICBpZiBsZW4oc2VsZi5fcmVjZW50X3NpZ3MpID4gMTI6CiAgICAgICAgICAgICAgICBzZWxmLl9yZWNlbnRfc2lncyA9IHNlbGYuX3JlY2VudF9zaWdzWy0xMjpdCgogICAgICAgICAgICBpZiBmcmFtZV9pbmZvWyJsZXZlbHNfY29tcGxldGVkIl0gIT0gc2VsZi5fbGFzdF9sZXZlbDoKICAgICAgICAgICAgICAgIHNlbGYuX2xhc3RfbGV2ZWwgPSBmcmFtZV9pbmZvWyJsZXZlbHNfY29tcGxldGVkIl0KICAgICAgICAgICAgICAgIGlmIHNlbGYuX2NvbnZlcnNhdGlvbjoKICAgICAgICAgICAgICAgICAgICBzZWxmLl9jb252ZXJzYXRpb24uYXBwZW5kKHsKICAgICAgICAgICAgICAgICAgICAgICAgInJvbGUiOiAidXNlciIsCiAgICAgICAgICAgICAgICAgICAgICAgICJjb250ZW50IjogZiJbTEVWRUwgQ09NUExFVEVEISBOb3cgb24gbGV2ZWwge2ZyYW1lX2luZm9bJ2xldmVsc19jb21wbGV0ZWQnXSArIDF9LiBSZS1hc3Nlc3MuXSIKICAgICAgICAgICAgICAgICAgICB9KQoKICAgICAgICAgICAgaWYgc2VsZi5faXNfbG9vcGVkKCk6CiAgICAgICAgICAgICAgICBhdmFpbGFibGUgPSBbYSBmb3IgYSBpbiBmcmFtZV9pbmZvWyJhdmFpbGFibGVfYWN0aW9ucyJdIGlmIGEgIT0gIlJFU0VUIl0KICAgICAgICAgICAgICAgIGlmIGF2YWlsYWJsZToKICAgICAgICAgICAgICAgICAgICBjaG9pY2UgPSByYW5kb20uY2hvaWNlKGF2YWlsYWJsZSkKICAgICAgICAgICAgICAgICAgICBzZWxmLl9yZWNlbnRfYWN0aW9ucy5hcHBlbmQoY2hvaWNlKQogICAgICAgICAgICAgICAgICAgIGlmIGxlbihzZWxmLl9yZWNlbnRfYWN0aW9ucykgPiAxMjoKICAgICAgICAgICAgICAgICAgICAgICAgc2VsZi5fcmVjZW50X2FjdGlvbnMgPSBzZWxmLl9yZWNlbnRfYWN0aW9uc1stMTI6XQogICAgICAgICAgICAgICAgICAgIHNlbGYuX3RvdGFsX2FjdGlvbnMgKz0gMQogICAgICAgICAgICAgICAgICAgIHNlbGYuX3VuaXF1ZV9zdGF0ZXMuYWRkKHNpZykKICAgICAgICAgICAgICAgICAgICBzZWxmLl9hY3Rpb25zX2luX2dhbWUgKz0gMQogICAgICAgICAgICAgICAgICAgIHJldHVybiB7Im5hbWUiOiBjaG9pY2UsICJyZWFzb25pbmciOiB7ImFudGlfbG9vcCI6IFRydWV9fQoKICAgICAgICAgICAgYWN0aW9uX2RpY3QgPSBzZWxmLl9jYWxsX2xsbShmcmFtZV9pbmZvKQogICAgICAgICAgICBzZWxmLl90b3RhbF9hY3Rpb25zICs9IDEKICAgICAgICAgICAgc2VsZi5fdW5pcXVlX3N0YXRlcy5hZGQoc2lnKQogICAgICAgICAgICBzZWxmLl9yZWNlbnRfYWN0aW9ucy5hcHBlbmQoYWN0aW9uX2RpY3QuZ2V0KCJuYW1lIiwgIlJFU0VUIikpCiAgICAgICAgICAgIGlmIGxlbihzZWxmLl9yZWNlbnRfYWN0aW9ucykgPiAxMjoKICAgICAgICAgICAgICAgIHNlbGYuX3JlY2VudF9hY3Rpb25zID0gc2VsZi5fcmVjZW50X2FjdGlvbnNbLTEyOl0KICAgICAgICAgICAgc2VsZi5fYWN0aW9uc19pbl9nYW1lICs9IDEKCiAgICAgICAgICAgIHJldHVybiBzZWxmLl90b19nYW1lX2FjdGlvbihhY3Rpb25fZGljdCkKCiAgICAgICAgZXhjZXB0IEV4Y2VwdGlvbiBhcyBlOgogICAgICAgICAgICBpbXBvcnQgdHJhY2ViYWNrCiAgICAgICAgICAgIHByaW50KGYiW0xMTUFnZW50XSBFcnJvcjoge2V9IikKICAgICAgICAgICAgdHJhY2ViYWNrLnByaW50X2V4YygpCiAgICAgICAgICAgIGFjdGlvbiA9IEdhbWVBY3Rpb24uUkVTRVQKICAgICAgICAgICAgYWN0aW9uLnNldF9kYXRhKHt9KQogICAgICAgICAgICBhY3Rpb24ucmVhc29uaW5nID0geyJlcnJvciI6IHN0cihlKX0KICAgICAgICAgICAgcmV0dXJuIGFjdGlvbgogICAgICAgIGZpbmFsbHk6CiAgICAgICAgICAgIHRyeToKICAgICAgICAgICAgICAgIHNlbGYuX0FDVElPTl9MT0NLLnJlbGVhc2UoKQogICAgICAgICAgICBleGNlcHQgUnVudGltZUVycm9yOgogICAgICAgICAgICAgICAgcGFzcwoKICAgIGRlZiB0YWtlX2FjdGlvbihzZWxmLCBhY3Rpb24pOgogICAgICAgIHRyeToKICAgICAgICAgICAgZnJhbWUgPSBzdXBlcigpLnRha2VfYWN0aW9uKGFjdGlvbikKICAgICAgICAgICAgaWYgZnJhbWUgaXMgbm90IE5vbmU6CiAgICAgICAgICAgICAgICBmcmFtZS5hY3Rpb25faW5wdXQgPSBBY3Rpb25JbnB1dCgKICAgICAgICAgICAgICAgICAgICBpZD1hY3Rpb24sCiAgICAgICAgICAgICAgICAgICAgZGF0YT1hY3Rpb24uYWN0aW9uX2RhdGEubW9kZWxfZHVtcCgpLAogICAgICAgICAgICAgICAgICAgIHJlYXNvbmluZz1nZXRhdHRyKGFjdGlvbiwgInJlYXNvbmluZyIsIE5vbmUpLAogICAgICAgICAgICAgICAgKQogICAgICAgICAgICByZXR1cm4gZnJhbWUKICAgICAgICBmaW5hbGx5OgogICAgICAgICAgICBwYXNzCgogICAgZGVmIGZpbmFsaXplX2RpYWdub3N0aWNzKHNlbGYpOgogICAgICAgIGVsYXBzZWQgPSB0aW1lLnRpbWUoKSAtIHNlbGYuX2dhbWVfc3RhcnQKICAgICAgICBwcmludChmIltMTE1BZ2VudF0gR2FtZSBkb25lOiB7c2VsZi5fYWN0aW9uc19pbl9nYW1lfSBhY3Rpb25zIGluIHtlbGFwc2VkOi4wZn1zLCAiCiAgICAgICAgICAgICAgZiJ7bGVuKHNlbGYuX3VuaXF1ZV9zdGF0ZXMpfSB1bmlxdWUgc3RhdGVzLCB7c2VsZi5fZ2FtZV9vdmVyX2NvdW50fSBkZWF0aHMiKQo="
agent_code = base64.b64decode(agent_b64).decode()
agent_dir = Path("/tmp/arc_llm_t4")
agent_dir.mkdir(parents=True, exist_ok=True)
(agent_dir / "__init__.py").write_text("")
(agent_dir / "llm_agent.py").write_text(agent_code)
print(f"Agent deployed to {agent_dir} ({len(agent_code)} chars)")
# Verify it compiles
compile(agent_code, "llm_agent.py", "exec")
print("Agent code compiles OK")

In [ ]:
import torch
import shutil
import urllib.request

if TRUE_SUBMISSION:
    framework_src = Path("/kaggle/input/competitions/arc-prize-2026-arc-agi-3/ARC-AGI-3-Agents")
    framework = WORKING / "ARC-AGI-3-Agents"
    if framework.exists():
        shutil.rmtree(framework)
    shutil.copytree(framework_src, framework)

    target = framework / "agents" / "templates"
    target.mkdir(parents=True, exist_ok=True)
    shutil.copy2(Path("/tmp/arc_llm_t4/llm_agent.py"), target / "llm_agent.py")
    (target / "__init__.py").write_text("")

    # Register agent
    NEWLINE = chr(10)
    init_py = NEWLINE.join([
        '"""Agent registry."""',
        'from typing import Type',
        'from dotenv import load_dotenv',
        'from .agent import Agent, Playback',
        'from .recorder import Recorder',
        'from .swarm import Swarm',
        'from .templates.random_agent import Random',
        'from .templates.my_agent import MyAgent',
        'from .templates.llm_agent import LLMAgent',
        '',
        'load_dotenv()',
        'AVAILABLE_AGENTS = {"random": Random, "myagent": MyAgent, "llmagent": LLMAgent}',
        'for recording in Recorder.list():',
        '    AVAILABLE_AGENTS[recording] = Playback',
    ]) + NEWLINE
    (framework / "agents" / "__init__.py").write_text(init_py)

    env_content = "\n".join([
        "SCHEME=http", "HOST=gateway", "PORT=8001",
        "ARC_API_KEY=competition-gateway",
        "ARC_BASE_URL=http://gateway:8001/",
        "OPERATION_MODE=online", "ENVIRONMENTS_DIR=",
        f"RECORDINGS_DIR={WORKING}/recordings",
    ]) + "\n"
    (framework / ".env").write_text(env_content)

    # Set model on agent class
    sys.path.insert(0, str(target))
    from llm_agent import LLMAgent as AgentClass
    AgentClass.set_model(model, tokenizer)

    # Wait for gateway
    print("Waiting for gateway...")
    deadline = time.time() + 600
    while time.time() < deadline:
        try:
            with urllib.request.urlopen("http://gateway:8001/api/games", timeout=10) as r:
                if r.status < 500:
                    break
        except Exception:
            pass
        time.sleep(5)

    run_env = os.environ.copy()
    run_env["MPLBACKEND"] = "Agg"
    run_env["PYDEVD_DISABLE_FILE_VALIDATION"] = "1"

    print("Starting LLM agent run...")
    result = subprocess.run(
        [sys.executable, "main.py", "--agent", "llmagent"],
        cwd=framework, env=run_env,
        capture_output=True, text=True, timeout=32000,
    )
    print("STDOUT:", result.stdout[-3000:] if result.stdout else "(empty)")
    print("STDERR:", result.stderr[-3000:] if result.stderr else "(empty)")
    print(f"Return code: {result.returncode}")
else:
    import pandas as pd
    placeholder = pd.DataFrame(
        [["placeholder_0", "placeholder", True, 0.0]],
        columns=["row_id", "game_id", "end_of_game", "score"],
    )
    placeholder.to_parquet(WORKING / "submission.parquet", index=False)
    print("Wrote save-phase placeholder submission.parquet")

In [ ]:
elapsed = time.time() - NOTEBOOK_START
print(f"Notebook elapsed: {elapsed:.0f}s ({elapsed/60:.1f} min)")